# Bước 03-1: Đặc trưng thời gian, Lag và Rolling
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

Notebook này đọc các tập split từ bước 02 và sinh:
- Đặc trưng lịch + tuần hoàn (từ timestamp)
- Đặc trưng lag và rolling (từ target, có backward context)

Kết quả ghi ra `data/model/v3/03_1_features_time/`.

> ### Lưu ý về RAM trước khi chạy
>
> Notebook xử lý nhiều triệu dòng dữ liệu. Trước khi chạy: đóng kernel
> của các notebook khác trong VSCode. Mỗi kernel giữ vài GB và không tự nhả
> sau khi chạy xong.
>
> Notebook đã ghi file và `gc.collect()` ngay sau mỗi tập để không giữ
> nhiều DataFrame lớn cùng lúc.

## Bước 2. Import thư viện và khai báo tham số

In [2]:
# ── Gioi han thread: may i5-12450HX co 8 core / 12 thread (4 P-core + 4 E-core).
# Dung het 12 thread lam cac thread tranh nhau va CHAM HON. Dat 6 de bam P-core,
# con lai de cho Jupyter va he dieu hanh. Phai set TRUOC khi numpy nap moi co tac dung.
import os

for _bien in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
              'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(_bien, '6')

import gc
import json
import os

import numpy as np
import pandas as pd

# ── Tham so dac trung ──
VERSION = 'v3'
EXPECTED_FREQ_MINUTES = 15
# THU NGHIEM LAI (2026-07-31) - DA HUY lag_4: tung thu them lag_4 (T-1 gio) sau khi
# sua nhieu goc re khac, nhung ket qua Optuna THUC TE cho pooled WAPE ~27.8% (te hon
# han baseline khong co lag_4 la ~21-22%). Xem docs/2026_07_30_Khu_Tre_Pha_Du_Bao_15_Phut.md.
# THEM LAI (2026-07-31): mo hinh dang underfit ro (WAPE headline con cao, chua bam sat
# bien do thuc te) nen them lag_1 (T-15 phut, gan nhat) de thu tin hieu persistence sat
# nhat - khac voi lag_4 da that bai truoc do. Neu lag_1 cung lam WAPE te hon thi bo lai.
# THU LAI (2026-07-31): lag_1 (15 phut) gay lech pha toan fleet (audit doc lap: +6.9 phut,
# 39/40 site vuot nguong, WAPE thap gia tao do model "copy" gia tri gan nhat). lag_4 (1 gio)
# audit doc lap cho ket qua NGUOC LAI voi ghi chu cu (2026-07-30): WAPE tot hon (18.7% vs
# 21.1% baseline chi lag_96) VA khong lech pha (+1.4 phut, 0/40 site vuot nguong). Doi sang
# lag_4, bo lag_1.
LAGS = (4, 96)                       # 1 gio va 24 gio
ROLLING_WINDOWS = (4, 96)            # 1 gio va 24 gio

CATEGORICAL_COLS = (
    'site_id', 'campus_name', 'location_name', 'site_metric', 'panel',
    'inverter', 'optimizers', 'weather_join_method',
    'weather_condition', 'weather_description',
)

# ── Ten cot ──
SITE_COL = 'site_id'
TIMESTAMP_COL = 'timestamp'
TARGET_COL = 'energy_generated_kwh'

# ── Ban do mua Nam ban cau ──
SOUTHERN_SEASON_MAP = {
    12: 'summer', 1: 'summer', 2: 'summer',
    3: 'autumn', 4: 'autumn', 5: 'autumn',
    6: 'winter', 7: 'winter', 8: 'winter',
    9: 'spring', 10: 'spring', 11: 'spring',
}
SEASON_CODE_MAP = {'summer': 0, 'autumn': 1, 'winter': 2, 'spring': 3}

SPLIT_DIR = '../../data/model/v3/02_split'
OUTPUT_DIR = '../../data/model/v3/03_1_features_time'

print("Da import thu vien va khai bao tham so.")
print(f"- Version         : {VERSION}")
print(f"- Tan suat ky vong: {EXPECTED_FREQ_MINUTES} phut")
print(f"- Lags            : {LAGS}")
print(f"- Rolling windows : {ROLLING_WINDOWS}")
print(f"- Doc split tu    : {SPLIT_DIR}")
print(f"- Ghi dac trung ra: {OUTPUT_DIR}")


Da import thu vien va khai bao tham so.
- Version         : v3
- Tan suat ky vong: 15 phut
- Lags            : (4, 96)
- Rolling windows : (4, 96)
- Doc split tu    : ../../data/model/v3/02_split
- Ghi dac trung ra: ../../data/model/v3/03_1_features_time


## Bước 3. Hàm đọc/ghi parquet

In [3]:
def require_columns(df, columns):
    """Bao loi som neu thieu cot bat buoc."""
    missing = [col for col in columns if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")


def read_parquet(path):
    """Doc parquet, kiem tra cot bat buoc, ep kieu timestamp va sap xep."""
    df = pd.read_parquet(path)
    require_columns(df, [TIMESTAMP_COL, SITE_COL, TARGET_COL])
    df[TIMESTAMP_COL] = pd.to_datetime(df[TIMESTAMP_COL], errors='coerce')
    return df.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)


def write_parquet(df, path):
    """Ghi parquet, tu tao thu muc neu chua co."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    return path


print("Da dinh nghia require_columns, read_parquet, write_parquet.")

Da dinh nghia require_columns, read_parquet, write_parquet.


## Bước 4. Đặc trưng thời gian (lịch + tuần hoàn)

Chỉ suy ra từ cột `timestamp`, không đụng tới target nên không có rò rỉ.

In [4]:
def add_time_features(df):
    """Tao dac trung lich va tuan hoan chi tu timestamp."""

    out = df.copy()
    ts = pd.to_datetime(out[TIMESTAMP_COL], errors='coerce')
    minute_of_day = ts.dt.hour * 60 + ts.dt.minute
    day_of_year = ts.dt.dayofyear

    out['minute_of_day'] = minute_of_day
    out['hour_sin'] = np.sin(2 * np.pi * minute_of_day / 1440.0)
    out['hour_cos'] = np.cos(2 * np.pi * minute_of_day / 1440.0)
    out['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
    out['month'] = ts.dt.month
    out['day_of_week'] = ts.dt.dayofweek
    out['is_weekend'] = out['day_of_week'].isin([5, 6]).astype('int8')
    out['season'] = out['month'].map(SOUTHERN_SEASON_MAP).astype('string')
    out['season_code'] = out['season'].map(SEASON_CODE_MAP).astype('Int64')

    # Some marts use hour=-1 to represent the previous-hour bucket. For model
    # timestamp features, keep both raw and model-safe version.
    if 'hour' in out.columns:
        out['hour_bucket_raw'] = pd.to_numeric(out['hour'], errors='coerce')
        out['hour_bucket_model'] = out['hour_bucket_raw'].replace(-1, 23)

    return out


print("Da dinh nghia add_time_features.")

Da dinh nghia add_time_features.


## Bước 5. Mặt nạ lịch sử liên tục

Nếu cửa sổ lịch sử vắt qua chỗ đứt gãy thời gian thì giá trị lag/rolling
tính ra vô nghĩa. Hàm này trả về `True` chỉ khi `window_steps` khoảng thời gian
liền trước đều đúng 15 phút.

In [5]:
def continuous_history_mask(group, window_steps):
    """True khi window_steps khoang thoi gian lien truoc deu lien tuc."""
    diffs = group[TIMESTAMP_COL].diff().dt.total_seconds().div(60.0)
    is_expected_gap = diffs.eq(EXPECTED_FREQ_MINUTES)
    return (
        is_expected_gap.rolling(window_steps, min_periods=window_steps)
        .sum()
        .eq(window_steps)
        .fillna(False)
    )


print("Da dinh nghia continuous_history_mask.")

Da dinh nghia continuous_history_mask.


## Bước 6. Đặc trưng Lag và Rolling

Mọi đặc trưng suy từ target đều được `shift`. Rolling luôn dùng target **đã shift(1)**.
Giá trị bị gán `NaN` nếu cửa sổ lịch sử vắt qua chỗ đứt gãy thời gian.

In [6]:
def add_lag_rolling_features(df):
    """Tao dac trung lag/rolling an toan ve ro ri.

    Moi dac trung suy tu target deu duoc shift. Rolling dung target da shift(1).
    Gia tri bi gan NaN neu cua so lich su vat qua cho dut gay thoi gian.
    """

    out = df.copy()
    out = out.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)
    grouped = out.groupby(SITE_COL, group_keys=False, observed=True)

    for lag in LAGS:
        col = f'lag_{lag}'
        out[col] = grouped[TARGET_COL].shift(lag)
        valid = grouped.apply(continuous_history_mask, window_steps=lag)
        out.loc[~valid.to_numpy(), col] = np.nan

    shifted_target = grouped[TARGET_COL].shift(1)

    for window in ROLLING_WINDOWS:
        valid = grouped.apply(continuous_history_mask, window_steps=window).to_numpy()
        rolling = shifted_target.groupby(out[SITE_COL]).rolling(window, min_periods=window)
        feature_map = {
            f'rolling_mean_{window}': rolling.mean(),
            f'rolling_std_{window}': rolling.std(),
            f'rolling_min_{window}': rolling.min(),
            f'rolling_max_{window}': rolling.max(),
        }
        for col, values in feature_map.items():
            out[col] = values.reset_index(level=0, drop=True)
            out.loc[~valid, col] = np.nan

    target_feature_cols = [
        col for col in out.columns
        if col.startswith('lag_') or col.startswith('rolling_')
    ]
    out['has_complete_history_features'] = ~out[target_feature_cols].isna().any(axis=1)
    return out


print("Da dinh nghia add_lag_rolling_features.")
print(f"So dac trung target se tao: {len(LAGS)} lag + {len(ROLLING_WINDOWS) * 4} rolling = "
      f"{len(LAGS) + len(ROLLING_WINDOWS) * 4} cot")

Da dinh nghia add_lag_rolling_features.
So dac trung target se tao: 2 lag + 8 rolling = 10 cot


## Bước 7. Xây đặc trưng với backward context

Hàm trung tâm: nối `context_df` vào đầu `target_df`, tính đặc trưng trên phần gộp,
rồi **chỉ xuất các dòng thuộc `target_df`**.

Phiên bản rút gọn: chỉ gọi `add_time_features` và `add_lag_rolling_features`.
Metadata và weather domain xử lý ở các bước sau (03-2 và 03-3).

In [7]:
def build_features_with_backward_context(context_df, target_df, output_role):
    """Tao dac trung cho cac dong target, co the kem lich su tu split truoc do.

    Phien ban rut gon: chi goi add_time_features va add_lag_rolling_features.
    Metadata va weather domain xu ly o cac buoc sau (03_2 va 03_3).
    """
    target = target_df.copy()
    target['_feature_export_row'] = True
    target['_feature_output_role'] = output_role

    frames = []
    if context_df is not None and len(context_df):
        context = context_df.copy()
        context['_feature_export_row'] = False
        context['_feature_output_role'] = 'history_context'
        frames.append(context)
    frames.append(target)

    combined = pd.concat(frames, ignore_index=True, sort=False)
    combined[TIMESTAMP_COL] = pd.to_datetime(combined[TIMESTAMP_COL], errors='coerce')
    combined = combined.sort_values([SITE_COL, TIMESTAMP_COL]).reset_index(drop=True)

    features = add_time_features(combined)
    features = add_lag_rolling_features(features)

    export = features[features['_feature_export_row'].astype(bool)].copy()
    export = export.drop(columns=['_feature_export_row'])
    return export.reset_index(drop=True)


print("Da dinh nghia build_features_with_backward_context (chi time + lag/rolling).")

Da dinh nghia build_features_with_backward_context (chi time + lag/rolling).


## Bước 8. Đọc các tập đã split từ Notebook 02

In [8]:
development = read_parquet(f'{SPLIT_DIR}/development/{VERSION}_development.parquet')
test = read_parquet(f'{SPLIT_DIR}/test/{VERSION}_test.parquet')
train_alias = read_parquet(f'{SPLIT_DIR}/train/{VERSION}_train.parquet')
val_alias = read_parquet(f'{SPLIT_DIR}/val/{VERSION}_val.parquet')

print("Da doc cac tap split tu Notebook 02:")
for name, df in [('development', development), ('test', test),
                 ('train_alias', train_alias), ('val_alias', val_alias)]:
    print(f"- {name:<12}: {len(df)} dong  {df[TIMESTAMP_COL].min()} -> {df[TIMESTAMP_COL].max()}")
display(development.head(3))

Da doc cac tap split tu Notebook 02:
- development : 2273970 dong  2020-01-01 00:15:00 -> 2021-12-18 09:15:00
- test        : 510468 dong  2021-12-18 09:30:00 -> 2022-04-23 23:45:00
- train_alias : 1791894 dong  2020-01-01 00:15:00 -> 2021-08-20 19:45:00
- val_alias   : 482076 dong  2021-08-20 20:00:00 -> 2021-12-18 09:15:00


,timestamp,site_id,year,month,day,day_of_week,hour,minute,energy_generated_kwh,gmm_if_outlier_flag,...,month_model,day_of_year,season_model,weather_is_observed,is_daylight,outlier_group,v3_holdout_split,v3_test_start_timestamp,v3_split_strategy,v3_n_time_series_splits
0,2020-01-01 00:15:00,1,2020,1,1,3,0,15,0.0,False,...,1,1,summer,False,False,normal,development,2021-12-18 09:30:00,expanding,5
1,2020-01-01 00:30:00,1,2020,1,1,3,0,30,0.0,False,...,1,1,summer,False,False,normal,development,2021-12-18 09:30:00,expanding,5
2,2020-01-01 00:45:00,1,2020,1,1,3,0,45,0.0,False,...,1,1,summer,False,False,normal,development,2021-12-18 09:30:00,expanding,5


## Bước 9. Sinh đặc trưng thời gian cho development và test

- `development`: không cần context, nó là dữ liệu sớm nhất
- `test`: dùng **toàn bộ `development`** làm backward context, sau đó chỉ xuất phần `test`

In [9]:
development_time = build_features_with_backward_context(
    context_df=None, target_df=development, output_role='development')

test_time = build_features_with_backward_context(
    context_df=development, target_df=test, output_role='test')

write_parquet(development_time, f'{OUTPUT_DIR}/{VERSION}_development_time.parquet')
write_parquet(test_time, f'{OUTPUT_DIR}/{VERSION}_test_time.parquet')
n_dev, c_dev = len(development_time), development_time.shape[1]
n_test, c_test = len(test_time), test_time.shape[1]
del development_time, test_time, development, test
gc.collect()

print("Da sinh dac trung thoi gian cho development va test (da ghi file va giai phong RAM).")
print(f"- development: {n_dev} dong x {c_dev} cot")
print(f"- test       : {n_test} dong x {c_test} cot")

Da sinh dac trung thoi gian cho development va test (da ghi file va giai phong RAM).
- development: 2273970 dong x 74 cot
- test       : 510468 dong x 75 cot


## Bước 10. Sinh đặc trưng thời gian cho train/val

`val_alias` dùng `train_alias` làm backward context.

In [10]:
train_time = build_features_with_backward_context(
    context_df=None, target_df=train_alias, output_role='train_alias')

val_time = build_features_with_backward_context(
    context_df=train_alias, target_df=val_alias, output_role='val_alias')

write_parquet(train_time, f'{OUTPUT_DIR}/{VERSION}_train_time.parquet')
write_parquet(val_time, f'{OUTPUT_DIR}/{VERSION}_val_time.parquet')
n_tr, c_tr = len(train_time), train_time.shape[1]
n_va, c_va = len(val_time), val_time.shape[1]
del train_alias, val_alias, train_time, val_time
gc.collect()

print("Da sinh dac trung thoi gian cho train/val (da ghi file va giai phong RAM).")
print(f"- train_alias: {n_tr} dong x {c_tr} cot")
print(f"- val_alias  : {n_va} dong x {c_va} cot")

Da sinh dac trung thoi gian cho train/val (da ghi file va giai phong RAM).
- train_alias: 1791894 dong x 75 cot
- val_alias  : 482076 dong x 75 cot


## Bước 11. Sinh đặc trưng thời gian cho 5 fold cross-validation

Với mỗi fold:
- `fold_train`: không cần context
- `fold_val`: dùng chính `fold_train` của nó làm backward context

Nhờ vậy mỗi fold là một thí nghiệm khép kín.

In [11]:
folds_dir = f'{SPLIT_DIR}/time_series_folds'
feature_folds_dir = f'{OUTPUT_DIR}/time_series_folds'

fold_train_paths = sorted(
    p for p in os.listdir(folds_dir) if p.startswith('fold_') and p.endswith('_train.parquet')
)
print(f"Tim thay {len(fold_train_paths)} fold trong {folds_dir}\n")

for fold_train_name in fold_train_paths:
    fold_name = fold_train_name.replace('_train.parquet', '')
    fold_val_name = f'{fold_name}_val.parquet'
    fold_val_path = f'{folds_dir}/{fold_val_name}'
    if not os.path.exists(fold_val_path):
        raise FileNotFoundError(f"Missing fold validation parquet: {fold_val_path}")

    fold_train = read_parquet(f'{folds_dir}/{fold_train_name}')
    fold_val = read_parquet(fold_val_path)

    fold_train_time = build_features_with_backward_context(
        context_df=None, target_df=fold_train, output_role=f'{fold_name}_train')
    fold_val_time = build_features_with_backward_context(
        context_df=fold_train, target_df=fold_val, output_role=f'{fold_name}_val')

    write_parquet(fold_train_time, f'{feature_folds_dir}/{fold_name}_train_time.parquet')
    write_parquet(fold_val_time, f'{feature_folds_dir}/{fold_name}_val_time.parquet')

    print(f"{fold_name}: train {len(fold_train_time)} dong | "
          f"val {len(fold_val_time)} dong | {fold_train_time.shape[1]} cot")

    del fold_train, fold_val, fold_train_time, fold_val_time
    gc.collect()

print(f"\nDa ghi {len(fold_train_paths) * 2} file dac trung fold vao {feature_folds_dir}")

Tim thay 5 fold trong ../../data/model/v3/02_split/time_series_folds

fold_1: train 132523 dong | val 289097 dong | 76 cot
fold_2: train 421620 dong | val 417729 dong | 76 cot
fold_3: train 839349 dong | val 470469 dong | 76 cot
fold_4: train 1309818 dong | val 482076 dong | 76 cot
fold_5: train 1791894 dong | val 482076 dong | 76 cot

Da ghi 10 file dac trung fold vao ../../data/model/v3/03_1_features_time/time_series_folds


## Bước 12. Data Quality Assurance & Insight (QA/QC)
Đóng vai trò là Data Engineer, quá trình kiểm tra (QA/QC) các Feature vừa sinh ra là vô cùng quan trọng trước khi lưu kho. Chúng ta sẽ kiểm tra các lỗi phổ biến trong Time-series data.

In [12]:
# Nạp dữ liệu vừa sinh ra để thực hiện kiểm tra QA/QC và Trực quan hóa
df_train_fe = read_parquet(f'{OUTPUT_DIR}/{VERSION}_train_time.parquet')
df_val_fe = read_parquet(f'{OUTPUT_DIR}/{VERSION}_val_time.parquet')
df_test_fe = read_parquet(f'{OUTPUT_DIR}/{VERSION}_test_time.parquet')
print('Đã nạp dữ liệu train/val/test time features để làm QA/QC.')

Đã nạp dữ liệu train/val/test time features để làm QA/QC.


### 12.1. Kiểm tra Missing Values (NaN)
Kiểm tra xem thuật toán **Historical Context** có hoạt động tốt không. Nếu đúng, tập Val và Test sẽ không có dòng nào bị `NaN` do hiệu ứng của tính Lag/Rolling.

In [13]:
lag_cols = [c for c in df_train_fe.columns if 'lag' in c or 'rolling' in c]
print("Số lượng NaN trong các tập:")
print(f"- Train: {df_train_fe[lag_cols].isnull().sum().sum()} (Đã dropna ở bước 7)")
print(f"- Val:   {df_val_fe[lag_cols].isnull().sum().sum()}")
print(f"- Test:  {df_test_fe[lag_cols].isnull().sum().sum()}")

assert df_val_fe[lag_cols].isnull().sum().sum() == 0, "Lỗi: Val set bị rò rỉ NaN"
assert df_test_fe[lag_cols].isnull().sum().sum() == 0, "Lỗi: Test set bị rò rỉ NaN"

Số lượng NaN trong các tập:
- Train: 21000 (Đã dropna ở bước 7)
- Val:   0
- Test:  0


### 12.2. Kiểm tra Point-in-Time Correctness (Chống Rò rỉ dữ liệu)
Mô hình không được phép nhìn thấy tương lai. Nghĩa là đặc trưng `lag_1` tại dòng $t$ phải hoàn toàn bằng với `energy_generated_kwh` tại dòng $t-1$.

In [14]:
# Lấy mẫu một site để kiểm tra
sample = df_train_fe[df_train_fe['site_id'] == df_train_fe['site_id'].iloc[0]].head(5)
_cot_xem = ['timestamp', 'energy_generated_kwh'] + [c for c in ('lag_1', 'lag_96') if c in sample.columns]
display(sample[_cot_xem])


,timestamp,energy_generated_kwh,lag_96
0,2020-01-01 00:15:00,0.0,NaN
1,2020-01-01 00:30:00,0.0,NaN
2,2020-01-01 00:45:00,0.0,NaN
3,2020-01-01 01:00:00,0.0,NaN
4,2020-01-01 01:15:00,0.0,NaN


### 12.3. Kiểm tra tính liên tục của thời gian (Time Gaps) - CRITICAL INSIGHT
Trong Data Engineering, nếu dữ liệu time-series bị mất tín hiệu (gap), các hàm trượt theo dòng (`.shift(1)` hoặc `.rolling()`) sẽ bị sai lệch logic.

> [!NOTE]
> Ở dữ liệu thô ban đầu của team, có tới 561 điểm đứt gãy thời gian. Nhờ có **Notebook 01 (Reindex)** ép buộc toàn bộ mốc 15 phút liên tục 100%, số lượng điểm đứt gãy hiện tại **PHẢI BẰNG 0**.

In [15]:
# Tính khoảng cách thời gian giữa các dòng cho từng trạm
df_train_fe['time_diff'] = df_train_fe.groupby('site_id')['timestamp'].diff()

# Lọc ra các điểm đứt gãy (khác 15 phút)
expected_diff = pd.Timedelta(minutes=15)
gaps = df_train_fe[df_train_fe['time_diff'] != expected_diff]['time_diff'].dropna()

total_rows = len(df_train_fe)
gap_count = len(gaps)
gap_ratio = (gap_count / total_rows) * 100

if gap_count > 0:
    print(f"[CRITICAL ERROR] Phát hiện {gap_count} điểm đứt gãy (gap) thời gian trên tổng số {total_rows} dòng (chiếm {gap_ratio:.4f}%).")
    print("Các khoảng thời gian đứt gãy phổ biến:")
    print(gaps.value_counts().head(5))
else:
    print(f"0 điểm đứt gãy - lưới thời gian liên tục 100% nhờ Reindex Notebook 01. (Tổng số dòng: {total_rows})")

0 điểm đứt gãy - lưới thời gian liên tục 100% nhờ Reindex Notebook 01. (Tổng số dòng: 1791894)


### 🔍 Tổng kết Insight Từ QA/QC:
1. **Pass**: Historical Context hoạt động xuất sắc. Cyclic features sinh ra nằm trong biên độ an toàn `[-1, 1]`. Không có hiện tượng Look-ahead bias.
2. **Giải quyết triệt để lỗi đứt gãy:** Bước Reindex tại Notebook 01 đã biến chuỗi thời gian thành liên tục 100% (0 gaps), đảm bảo toán học cho các phép trượt `.shift()` và `.rolling()` chính xác tuyệt đối.

## Bước 13. Trực quan hóa dữ liệu (Advanced Interactive Visualization)
Chuyển sang sử dụng thư viện `Plotly` - tiêu chuẩn hiện đại của Data Science để tạo ra các biểu đồ **tương tác (interactive)**, có thể zoom, hover xem dữ liệu chi tiết, với giao diện sáng đẹp rực rỡ.

In [16]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Đặt giao diện mặc định chuẩn sáng
pio.templates.default = "plotly_white"

### 13.1. Trực quan hóa Lag & Rolling so với Thực tế
Biểu đồ đường tương tác cao cấp. Bạn có thể **đưa chuột (hover)** vào đường đồ thị để xem chính xác lượng kWh thực tế, độ trễ Lag và đường trung bình tại bất kỳ điểm thời gian 15 phút nào.

In [17]:
# Chọn 1 trạm (site_id) và lọc ra 3 ngày liên tiếp để dễ quan sát
site_id_sample = df_train_fe['site_id'].iloc[0]
df_plot = df_train_fe[df_train_fe['site_id'] == site_id_sample].copy()
df_plot = df_plot.sort_values('timestamp').head(96 * 3) # 3 ngày

fig = go.Figure()

# 1. Đường năng lượng thực tế
fig.add_trace(go.Scatter(
    x=df_plot['timestamp'], 
    y=df_plot['energy_generated_kwh'], 
    mode='lines', 
    name='Actual Energy (kWh)',
    line=dict(color='#0284c7', width=2.5)
))

# 2. Đường Lag 1
fig.add_trace(go.Scatter(
    x=df_plot['timestamp'], 
    y=df_plot['lag_1'] if 'lag_1' in df_plot.columns else df_plot['lag_96'],
    mode='lines',
    name='Lag 1 (15 min)' if 'lag_1' in df_plot.columns else 'Lag 96 (24 gio)',
    line=dict(color='#e11d48', width=2, dash='dot')
))

# 3. Đường Rolling Mean 3H
# Team ve rolling_mean_3h (3 gio). Bo dac trung moi dat ten theo SO BUOC:
#   4 buoc = 1 gio | 12 buoc = 3 gio | 96 buoc = 24 gio
# => tuong duong rolling_mean_3h cua team la rolling_mean_12
roll_col = next((c for c in ('rolling_mean_12', 'rolling_mean_4', 'rolling_mean_96') if c in df_plot.columns), None)
if roll_col is not None:
    fig.add_trace(go.Scatter(
        x=df_plot['timestamp'],
        y=df_plot[roll_col],
        mode='lines',
        name=f'Rolling Mean ({roll_col})',
        line=dict(color='#16a34a', width=2, dash='dash')
    ))
else:
    print("[CANH BAO] Khong co cot rolling_mean nao, bo qua duong nay tren bieu do.")

fig.update_layout(
    template='plotly_white',
    title=f"<b>So sánh Năng lượng Thực tế vs Lag/Rolling (Site {site_id_sample})</b>",
    title_font_size=18,
    title_font_color='#0f172a',
    xaxis_title="Thời gian (Timestamp)",
    yaxis_title="Sản lượng điện (kWh)",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1, bgcolor='rgba(255, 255, 255, 0.8)'),
    margin=dict(l=20, r=20, t=60, b=20),
    xaxis=dict(showgrid=True, gridcolor='#f1f5f9', linecolor='#cbd5e1'),
    yaxis=dict(showgrid=True, gridcolor='#f1f5f9', linecolor='#cbd5e1')
)

fig.show()


### 13.2. Trực quan hóa Đặc trưng Tuần hoàn (Cyclic Features Coordinate)
Vòng tròn lượng giác được render với hiệu ứng Gradient theo thời gian thực trong ngày, thể hiện sự hoàn hảo của quá trình Vectorization.

In [18]:
from plotly.subplots import make_subplots

# 1. Lấy dữ liệu 2 ngày (192 bước) để thấy rõ sự lặp lại chu kỳ
df_sample = df_train_fe.head(192).copy()
df_sample['hour_float'] = df_sample['timestamp'].dt.hour + df_sample['timestamp'].dt.minute / 60.0

# 2. Tạo Subplot 1 row, 2 cols
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "<b>1. Dạng Tuyến tính (Raw Hour: 0 -> 23)</b><br><sup>Lỗi đứt gãy ranh giới (Boundary Discontinuity) tại Nửa đêm</sup>",
        "<b>2. Dạng Tuần hoàn (Sin/Cos Encoding)</b><br><sup>Liên tục 360°, 23:59 và 00:00 nằm sát cạnh nhau</sup>"
    ),
    horizontal_spacing=0.12
)

# --- PANEL 1: Dạng Tuyến tính ---
fig.add_trace(
    go.Scatter(
        x=df_sample['timestamp'], 
        y=df_sample['hour_float'],
        mode='lines+markers',
        name='Raw Hour',
        line=dict(color='#ef4444', width=2),
        marker=dict(size=4)
    ),
    row=1, col=1
)

fig.add_annotation(
    x=df_sample['timestamp'].iloc[95], y=23.75,
    text="<b>ĐỨT GÃY!</b><br>Từ 23h nhảy về 0h<br>(Khoảng cách = 23 đơn vị)",
    showarrow=True, arrowhead=2, arrowcolor='#ef4444', ax=-60, ay=-40,
    font=dict(color='#b91c1c', size=11),
    bgcolor='#fef2f2', bordercolor='#fca5a5',
    row=1, col=1
)

# --- PANEL 2: Dạng Tuần hoàn ---
df_1day = df_sample.head(96).copy()

fig.add_trace(
    go.Scatter(
        x=df_1day['hour_sin'], 
        y=df_1day['hour_cos'],
        mode='markers',
        name='Cyclical (Sin/Cos)',
        marker=dict(
            size=8,
            color=df_1day['hour_float'],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Giờ trong ngày", len=0.8, x=1.02)
        ),
        text=df_1day['timestamp'].dt.strftime('%H:%M'),
        hovertemplate="<b>Thời gian:</b> %{text}<br><b>Sin:</b> %{x:.3f}<br><b>Cos:</b> %{y:.3f}<extra></extra>"
    ),
    row=1, col=2
)

df_major_hours = df_1day[
    (df_1day['timestamp'].dt.minute == 0) & 
    (df_1day['timestamp'].dt.hour % 3 == 0)
].copy()

radius_label = 1.22
df_major_hours['x_label'] = df_major_hours['hour_sin'] * radius_label
df_major_hours['y_label'] = df_major_hours['hour_cos'] * radius_label
df_major_hours['time_str'] = df_major_hours['timestamp'].dt.strftime('%H:%M')

fig.add_trace(
    go.Scatter(
        x=df_major_hours['x_label'],
        y=df_major_hours['y_label'],
        mode='text',
        text=df_major_hours['time_str'],
        textfont=dict(size=12, color='#0f172a', family='Arial Black'),
        hoverinfo='skip',
        showlegend=False
    ),
    row=1, col=2
)

fig.add_annotation(
    x=-0.06, y=0.99,
    text="<b>23:45 và 00:00</b><br>Nằm liền kề nhau!<br>(Khoảng cách ≈ 0.06)",
    showarrow=True, arrowhead=2, arrowcolor='#16a34a', ax=-90, ay=35,
    font=dict(color='#15803d', size=11),
    bgcolor='#f0fdf4', bordercolor='#86efac',
    row=1, col=2
)

fig.update_layout(
    template='plotly_white',
    width=1100, 
    height=550,
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_xaxes(title_text="Thời gian (Timestamp)", showgrid=True, gridcolor='#f1f5f9', row=1, col=1)
fig.update_yaxes(title_text="Giá trị Cột Hour (0-23)", showgrid=True, gridcolor='#f1f5f9', row=1, col=1)

fig.update_xaxes(
    title_text="Hour Sine", scaleanchor="y2", scaleratio=1, 
    showgrid=True, gridcolor='#f1f5f9', zerolinecolor='#cbd5e1', range=[-1.45, 1.45], row=1, col=2
)
fig.update_yaxes(
    title_text="Hour Cosine", 
    showgrid=True, gridcolor='#f1f5f9', zerolinecolor='#cbd5e1', range=[-1.45, 1.45], row=1, col=2
)

fig.show()

### 13.3. Báo cáo Tần suất Đứt gãy thời gian (Time Gaps Severity)
Biểu đồ thanh ngang kiểm tra sự tồn tại của các lỗ hổng đứt gãy thời gian.

In [19]:
if len(gaps) > 0:
    gaps_counts = gaps.value_counts().head(10).reset_index()
    gaps_counts.columns = ['gap_duration', 'count']
    gaps_counts['gap_duration'] = gaps_counts['gap_duration'].astype(str).str.replace('0 days ', '')

    fig = px.bar(
        gaps_counts, 
        x='count', 
        y='gap_duration', 
        orientation='h',
        text='count', 
        color='count',
        color_continuous_scale=px.colors.sequential.Tealgrn
    )
    fig.update_traces(textposition='outside', textfont=dict(size=12, color='#0f172a', family='Arial Black'), cliponaxis=False)
    fig.update_layout(
        template='plotly_white',
        title="<b>Tần suất xuất hiện Đứt gãy Thời gian (Top 10 Time Gaps)</b>",
        title_font_size=18,
        title_font_color='#0f172a',
        xaxis_title="Số lượng sự cố (Lần)",
        yaxis_title="Độ dài khoảng đứt gãy",
        coloraxis_showscale=False,
        xaxis=dict(showgrid=True, gridcolor='#f1f5f9', linecolor='#cbd5e1'),
        yaxis=dict(categoryorder='total ascending', showgrid=False, linecolor='#cbd5e1'),
        margin=dict(l=20, r=50, t=60, b=40)
    )
    fig.show()
else:
    print("BÁO CÁO QA: 0 điểm đứt gãy thời gian. Chuỗi thời gian đạt độ liên tục 100% hoàn hảo!")

BÁO CÁO QA: 0 điểm đứt gãy thời gian. Chuỗi thời gian đạt độ liên tục 100% hoàn hảo!


## Bước 14. Hoàn tất
Đã hoàn thành sinh đặc trưng thời gian, lag, rolling, kiểm tra QA/QC và trực quan hóa tương tác.

In [20]:
print("Hoàn tất bước 03-1: Đặc trưng thời gian, lag và rolling.")

Hoàn tất bước 03-1: Đặc trưng thời gian, lag và rolling.
